# Simple Authentication with Access Token (Standard OAuth 2.0)

This tutorial demonstrates the **standard OAuth 2.0 approach** using Access Tokens for basic authentication.

## When to Use Access Token

✅ **Use Access Token when:**
- Single-tenant application
- Only need basic user identity (sub, username)
- Don't need custom user attributes
- Following standard OAuth 2.0 patterns

❌ **Don't use Access Token when:**
- Multi-tenant application
- Need custom attributes (tenant_id, user_role)
- Need detailed user profile information

## What Access Token Contains (Cognito)

```json
{
  "sub": "user-uuid",              // ✓ User ID
  "username": "john",              // ✓ Username  
  "client_id": "3jea6...",         // ✓ Client ID
  "token_use": "access",
  "scope": "aws.cognito.signin.user.admin"
  // ❌ No aud field
  // ❌ No custom attributes
}
```

## Architecture

```
Client → Access Token → AgentCore → Strands Agent
                         (validates)  (extracts sub, username)
```

In [1]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents pyyaml

In [3]:
import boto3
import json
import time
import base64
import yaml

region = 'us-west-2'
cognito_client = boto3.client('cognito-idp', region_name=region)

print(f"Region: {region}")

Region: us-west-2


## Step 1: Setup Simple Cognito (No Custom Attributes)

In [4]:
user_pool_name = f"simple-auth-{int(time.time())}"

# Simple user pool - no custom attributes
user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={'PasswordPolicy': {'MinimumLength': 8}},
    Schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True}
    ],
    AutoVerifiedAttributes=['email']
)

user_pool_id = user_pool_response['UserPool']['Id']

app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"{user_pool_name}-client",
    GenerateSecret=False,
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH']
)

client_id = app_client_response['UserPoolClient']['ClientId']
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

cognito_config = {
    'user_pool_id': user_pool_id,
    'client_id': client_id,
    'discovery_url': discovery_url
}

print(f"✓ User Pool: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

✓ User Pool: us-west-2_gGRa9DTCG
✓ Client ID: 1gvaut718cr773gfvct6qt7m20


In [5]:
# Create simple test users
test_users = [
    {'username': 'alice', 'password': 'Pass123!', 'email': 'alice@example.com'},
    {'username': 'bob', 'password': 'Pass123!', 'email': 'bob@example.com'}
]

for user in test_users:
    try:
        cognito_client.admin_create_user(
            UserPoolId=user_pool_id,
            Username=user['username'],
            UserAttributes=[
                {'Name': 'email', 'Value': user['email']},
                {'Name': 'email_verified', 'Value': 'true'}
            ],
            TemporaryPassword=user['password'],
            MessageAction='SUPPRESS'
        )
        cognito_client.admin_set_user_password(
            UserPoolId=user_pool_id,
            Username=user['username'],
            Password=user['password'],
            Permanent=True
        )
        print(f"  ✓ {user['username']}")
    except cognito_client.exceptions.UsernameExistsException:
        print(f"  ↻ {user['username']} exists")

  ✓ alice
  ✓ bob


## Step 2: Create Simple Authentication Agent

In [6]:
%%writefile simple_auth_agent.py
from strands import Agent
import json
import base64
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

def decode_jwt(token):
    """Decode JWT payload"""
    try:
        parts = token.split('.')
        payload = parts[1]
        padding = 4 - (len(payload) % 4)
        if padding != 4:
            payload += '=' * padding
        return json.loads(base64.b64decode(payload))
    except:
        return None

model = BedrockModel(model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0")
agent = Agent(model=model, system_prompt="You're a helpful assistant.")

@app.entrypoint
def simple_auth_agent(payload):
    """Simple authentication - extract basic user info from Access Token"""
    
    # Get Authorization header
    headers = BedrockAgentCoreContext.get_request_headers() or {}
    auth = headers.get('authorization') or headers.get('Authorization')
    
    if not auth:
        return {
            "response": "Not authenticated",
            "user_info": {"authenticated": False}
        }
    
    # Decode Access Token
    token = auth.replace('Bearer ', '')
    claims = decode_jwt(token)
    
    if not claims:
        return {
            "response": "Invalid token",
            "user_info": {"authenticated": False}
        }
    
    # Extract basic info (what's available in Access Token)
    user_info = {
        "authenticated": True,
        "sub": claims.get('sub'),           # User ID
        "username": claims.get('username'),  # Username
        "client_id": claims.get('client_id'), # Client ID
        "token_use": claims.get('token_use')  # Should be 'access'
    }
    
    print(f"Authenticated user: {user_info['username']} (sub: {user_info['sub'][:20]}...)")
    
    # Process request
    user_input = payload.get("prompt", "")
    response = agent(user_input)
    
    return {
        "response": response.message['content'][0]['text'],
        "user_info": user_info
    }

if __name__ == "__main__":
    app.run()

Overwriting simple_auth_agent.py


## Step 3: Deploy with Access Token Configuration

In [7]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()

# Configure for Access Token (use allowedClients, not allowedAudience)
runtime.configure(
    entrypoint="simple_auth_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="simple_auth",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config['discovery_url'],
            "allowedClients": [cognito_config['client_id']]  # For Access Token
            # No allowedAudience - Access Token doesn't have aud field
        }
    }
)

print("✓ Configured for Access Token authentication")

Entrypoint parsed: file=/Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity1/simple_auth_agent.py, bedrock_agentcore_name=simple_auth_agent
Memory configured with STM only
Configuring BedrockAgentCore agent: simple_auth
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: simple_auth_mem-txKA6VDGNz
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity1/.dockerignore
Keeping 'simple_auth' as default agent
Bedrock AgentCore configured: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity1/.bedrock_agentcore.yaml


✓ Configured for Access Token authentication


In [8]:
# Add Authorization header forwarding
with open('.bedrock_agentcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['agents']['simple_auth']['request_header_configuration'] = {
    'requestHeaderAllowlist': ['Authorization']
}

with open('.bedrock_agentcore.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✓ Added Authorization header forwarding")

✓ Added Authorization header forwarding


In [9]:
print("Deploying agent...\n")
launch_result = runtime.launch(auto_update_on_conflict=True)
print(f"\n✓ Deployed: {launch_result.agent_arn}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: simple_auth
✅ MemoryManager initialized for region: us-west-2


Deploying agent...



🔎 Retrieving memory resource with ID: simple_auth_mem-txKA6VDGNz...
  Found memory: simple_auth_mem-txKA6VDGNz
Found existing memory in cloud: simple_auth_mem-txKA6VDGNz
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'simple_auth' to account 780622314958 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: simple_auth
✅ ECR repository available: 780622314958.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-simple_auth
Getting or creating execution role for agent: simple_auth
Using AWS region: us-west-2, account ID: 780622314958
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-d0278111c7


✅ Reusing existing ECR repository: 780622314958.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-simple_auth


✅ Reusing existing execution role: arn:aws:iam::780622314958:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-d0278111c7
✅ Execution role available: arn:aws:iam::780622314958:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-d0278111c7
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: simple_auth
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-d0278111c7
Reusing existing CodeBuild execution role: arn:aws:iam::780622314958:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-d0278111c7
Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: simple_auth/source.zip
Updated CodeBuild project: bedrock-agentcore-simple_auth-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.5s
🔄 DOWNLOAD_SOURCE started (total: 10s)
✅ DOWNLOAD_SOURCE co


✓ Deployed: arn:aws:bedrock-agentcore:us-west-2:780622314958:runtime/simple_auth-TrFcLPCtkU


In [10]:
print("Waiting for agent...")
status_response = runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED']:
    time.sleep(10)
    status_response = runtime.status()
    status = status_response.endpoint['status']
    print(f"  {status}")

if status == 'READY':
    print(f"\n✓ Agent ready")
else:
    raise Exception(f"Failed: {status}")

Waiting for agent...


✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: simple_auth_mem-txKA6VDGNz...
  Found memory: simple_auth_mem-txKA6VDGNz
Retrieved Bedrock AgentCore status for: simple_auth



✓ Agent ready


## Step 4: Test with Access Token

In [11]:
def get_tokens(username, password):
    response = cognito_client.initiate_auth(
        ClientId=cognito_config['client_id'],
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

# Authenticate Alice
alice_tokens = get_tokens('alice', 'Pass123!')
print("✓ Authenticated Alice")

✓ Authenticated Alice


In [12]:
# Inspect Access Token contents
def decode_token(token):
    parts = token.split('.')
    payload = parts[1]
    padding = 4 - (len(payload) % 4)
    if padding != 4:
        payload += '=' * padding
    return json.loads(base64.b64decode(payload))

access_claims = decode_token(alice_tokens['AccessToken'])

print("Access Token Contents:")
print("="*60)
print(f"  token_use: {access_claims.get('token_use')}")
print(f"  sub: {access_claims.get('sub')[:20]}...")
print(f"  username: {access_claims.get('username')}")
print(f"  client_id: {access_claims.get('client_id')}")
print(f"  scope: {access_claims.get('scope')}")
print(f"  aud: {access_claims.get('aud', 'NOT FOUND')}")
print("="*60)

Access Token Contents:
  token_use: access
  sub: 08219360-d001-707c-f...
  username: alice
  client_id: 1gvaut718cr773gfvct6qt7m20
  scope: aws.cognito.signin.user.admin
  aud: NOT FOUND


In [13]:
print("TEST: Access Token Authentication")
print("="*60)

# Use Access Token (standard OAuth 2.0)
access_token = alice_tokens['AccessToken']

response = runtime.invoke(
    {"prompt": "What is 5+3?"},
    bearer_token=access_token  # Access Token!
)

# Parse response
resp_data = response.get('response')
if isinstance(resp_data, str):
    try:
        parsed = json.loads(resp_data)
        agent_resp = parsed['response']
        user_info = parsed['user_info']
    except:
        agent_resp = resp_data
        user_info = response.get('user_info', {})
else:
    agent_resp = resp_data
    user_info = response.get('user_info', {})

print(f"\nAgent Response: {agent_resp}")
print(f"\nUser Info:")
print(f"  Authenticated: {user_info.get('authenticated')}")
print(f"  Username: {user_info.get('username')}")
print(f"  Sub: {user_info.get('sub', 'N/A')[:20]}...")
print(f"  Client ID: {user_info.get('client_id')}")
print(f"  Token Type: {user_info.get('token_use')}")

if user_info.get('authenticated') and user_info.get('token_use') == 'access':
    print("\n✓ SUCCESS: Access Token authentication working!")
    print("  Standard OAuth 2.0 flow confirmed")
else:
    print("\n✗ FAILED")

TEST: Access Token Authentication


✅ MemoryManager initialized for region: us-west-2
Memory is active, proceeding with invoke



Agent Response: 5 + 3 = 8

User Info:
  Authenticated: True
  Username: alice
  Sub: 08219360-d001-707c-f...
  Client ID: 1gvaut718cr773gfvct6qt7m20
  Token Type: access

✓ SUCCESS: Access Token authentication working!
  Standard OAuth 2.0 flow confirmed


### Test Multiple Users

In [14]:
print("TEST: Multiple Users")
print("="*60)

for user in test_users:
    tokens = get_tokens(user['username'], user['password'])
    access_token = tokens['AccessToken']
    
    response = runtime.invoke(
        {"prompt": "Hello"},
        bearer_token=access_token
    )
    
    resp_data = response.get('response')
    if isinstance(resp_data, str):
        try:
            parsed = json.loads(resp_data)
            user_info = parsed['user_info']
        except:
            user_info = response.get('user_info', {})
    else:
        user_info = response.get('user_info', {})
    
    print(f"\n{user['username']}:")
    print(f"  Authenticated: {user_info.get('authenticated')}")
    print(f"  Username: {user_info.get('username')}")
    print(f"  Token Type: {user_info.get('token_use')}")

TEST: Multiple Users

alice:
  Authenticated: True
  Username: alice
  Token Type: access

bob:
  Authenticated: True
  Username: bob
  Token Type: access


## Comparison: Access Token vs ID Token

### What We Can Extract

| Claim | Access Token | ID Token |
|-------|--------------|----------|
| `sub` | ✅ | ✅ |
| `username` | ✅ | ✅ |
| `client_id` | ✅ | ❌ |
| `aud` | ❌ | ✅ |
| `custom:tenant_id` | ❌ | ✅ |
| `custom:user_role` | ❌ | ✅ |

### AgentCore Configuration

```python
# Access Token
authorizer_configuration={
    "customJWTAuthorizer": {
        "discoveryUrl": "...",
        "allowedClients": [client_id]  # Validates client_id
    }
}

# ID Token
authorizer_configuration={
    "customJWTAuthorizer": {
        "discoveryUrl": "...",
        "allowedAudience": [client_id]  # Validates aud
    }
}
```

### Use Cases

**Use Access Token when:**
- ✅ Standard OAuth 2.0 compliance needed
- ✅ Single-tenant application
- ✅ Basic authentication (who is the user?)
- ✅ Scope-based authorization

**Use ID Token when:**
- ✅ Multi-tenant application
- ✅ Need custom user attributes
- ✅ Detailed user profile required
- ✅ No token introspection desired

## Summary

### What We Learned

1. **Access Token** is the standard OAuth 2.0 token for API authorization
2. Contains basic identity: `sub`, `username`, `client_id`
3. Does NOT contain custom attributes in Cognito
4. Use `allowedClients` for Access Token validation
5. Perfect for simple, single-tenant authentication

### Key Differences

| Aspect | This Example (Access Token) | Multi-Tenant (ID Token) |
|--------|----------------------------|------------------------|
| Token Type | Access | ID |
| Validation | allowedClients | allowedAudience |
| Custom Attributes | ❌ No | ✅ Yes |
| Use Case | Simple auth | Multi-tenant |
| OAuth 2.0 Standard | ✅ Yes | ❌ Non-standard |

### Congratulations!

You've implemented standard OAuth 2.0 authentication with Access Tokens in AgentCore! 🎉

In [15]:
# # Cleanup (optional)
# agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)
# agentcore_control.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
# cognito_client.delete_user_pool(UserPoolId=cognito_config['user_pool_id'])
# print("✓ Cleaned up")